# Lesson 4.3 · Metrics, threshold and imbalance

A classification model returns scores: turning them into decisions requires a **threshold**, and evaluating them requires a **metric**
suited to the cost of the errors. This lesson is the one you need to avoid being fooled by an accuracy of 95%.

**What you will learn**

- Confusion matrix, precision, recall and F1: what each one measures.
- ROC and precision–recall curves, and why the ROC AUC is not enough when the positive class is rare.
- How to choose the threshold (and why it is not 0.5) and what `class_weight` is for.

In [ ]:
import sys, pathlib
_root = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "common").is_dir())
sys.path.insert(0, str(_root))

import warnings; warnings.filterwarnings("ignore")     # clean output: here the warnings are just noise
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from common import datasets as ds
from common.verify import check, close, done, verify

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (average_precision_score, confusion_matrix, f1_score, precision_recall_curve, precision_score,
                             recall_score, roc_auc_score, roc_curve)
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

df = ds.loans(with_missing=False)
df["ratio"] = df["amount"] / df["income"]
cols = ["ratio", "age", "job_tenure", "income"]
X_tr, X_te, y_tr, y_te = train_test_split(df[cols], df["defaulted"], test_size=0.3, stratify=df["defaulted"], random_state=0)
model = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000)).fit(X_tr, y_tr)
scores = model.predict_proba(X_te)[:, 1]
print("defaulters in the test set:", int(y_te.sum()), "out of", len(y_te), f"({y_te.mean():.1%})")

## The confusion matrix and the derived metrics

With threshold 0.5 every example falls into one of four cells: true positives (TP), false positives (FP), false negatives (FN), true negatives (TN).

| metric | formula | answers |
|---|---|---|
| accuracy | (TP+TN) / all | how many did I get right? |
| **precision** | TP / (TP+FP) | among those I flagged, how many are really positive? |
| **recall** (sensitivity) | TP / (TP+FN) | among the true positives, how many did I find? |
| F1 | harmonic mean of precision and recall | a compromise in a single number |

In [ ]:
pred = (scores >= 0.5).astype(int)
tn, fp, fn, tp = confusion_matrix(y_te, pred).ravel()
print(f"TN={tn}  FP={fp}  FN={fn}  TP={tp}")
print(f"accuracy {(tp + tn) / len(y_te):.3f} | precision {precision_score(y_te, pred):.3f} | recall {recall_score(y_te, pred):.3f} | F1 {f1_score(y_te, pred):.3f}")
print(f"a model that always says 'no default' has accuracy {1 - y_te.mean():.3f}")

The model is only slightly less wrong than the trivial one, and finds a small fraction of the defaulters: with imbalanced classes **accuracy is
almost useless**. With threshold 0.5 the model is cautious: it declares default only whoever has a default probability above 50%, but the base rate is about 20%.

## The threshold is a business choice

Lowering the threshold finds more defaulters (recall goes up) but also flags more innocents (precision goes down). There is no right threshold: there is
the one that minimizes the cost, which must be chosen with the real costs (an unpaid loan costs much more than a customer wrongly turned down).

In [ ]:
rows = []
for s in [0.1, 0.2, 0.3, 0.4, 0.5]:
    p = (scores >= s).astype(int)
    rows.append({"threshold": s, "flagged": int(p.sum()), "precision": precision_score(y_te, p), "recall": recall_score(y_te, p), "F1": f1_score(y_te, p)})
pd.DataFrame(rows).round(3).set_index("threshold")

## ROC and precision–recall: all the thresholds together

The **ROC curve** shows recall (true positive rate) against false positive rate as the threshold varies; its area (AUC) is the probability that
a random positive gets a higher score than a random negative (0.5 = chance, 1 = perfect). The **precision–recall curve** shows how
the precision falls when you try to increase the recall: with rare classes it is more informative, because the ROC uses the false positive rate
(divided by the many negatives) and stays optimistic.

In [ ]:
fpr, tpr, _ = roc_curve(y_te, scores)
prec, rec, _ = precision_recall_curve(y_te, scores)
fig, ax = plt.subplots(1, 2, figsize=(9, 3.5))
ax[0].plot(fpr, tpr); ax[0].plot([0, 1], [0, 1], "k:"); ax[0].set(title=f"ROC (AUC = {roc_auc_score(y_te, scores):.3f})", xlabel="false positive rate", ylabel="recall")
ax[1].plot(rec, prec); ax[1].axhline(y_te.mean(), color="k", ls=":"); ax[1].set(title=f"PR (AP = {average_precision_score(y_te, scores):.3f})", xlabel="recall", ylabel="precision")
plt.tight_layout(); plt.show()

The dotted line in the PR plot is the precision of chance (the prevalence, about 0.2): it is the baseline to beat. In ROC the baseline is the diagonal.

## `class_weight`

`class_weight="balanced"` weighs the errors on the rare class more: it moves the model toward recall, it is almost equivalent to lowering the threshold.
**But the output probabilities are no longer calibrated**: "0.6" no longer means "6 out of 10 are positive". If you need true probabilities (to
compute an expected cost, for example) keep the model without weights and move the threshold, or recalibrate it (lesson 4.4).

In [ ]:
balanced = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000, class_weight="balanced")).fit(X_tr, y_tr)
for name, m in {"no weights": model, "balanced": balanced}.items():
    p = m.predict_proba(X_te)[:, 1]
    print(f"{name:10s} AUC {roc_auc_score(y_te, p):.3f} | recall at threshold 0.5: {recall_score(y_te, p >= 0.5):.3f} | mean predicted probability {p.mean():.3f} (true rate {y_te.mean():.3f})")

The AUC is practically identical (the ordering of the cases is the same), but with `balanced` the mean predicted probability is much higher than the real rate: the model is deliberately skewed.

> **Common mistake.** Optimizing the threshold on the test set. The threshold is a hyperparameter: it must be chosen with cross-validation on the training set (`cross_val_predict` with `method="predict_proba"`) and then applied to the test set once.

## Check your understanding

> *Click on a question to show its answer.*

<details><summary>I have 1 positive in 1000: which metric do I report?</summary>

Precision and recall at the chosen threshold, and the average precision (area under the PR curve). Accuracy is useless (99.9% by always saying no) and the ROC AUC can look high even when the precision is very low, because the false positives are divided by an enormous number of negatives.

</details>

<details><summary>Why does the AUC (almost) not change with `class_weight='balanced'` but the recall does?</summary>

The AUC depends only on the ordering of the scores; the weights change the scale of the scores (the probabilities shift upward) more than their order. The recall at threshold 0.5 depends instead on how many scores exceed 0.5, and that changes a lot.

</details>